# Playing a Looped Signal

It's often useful to be able to play a single signal back-to-back forever. Herein we'll demonstrate how to do this in Acadia. The hardware itself does not have an "infinite stream" mode; instead, the sequencer is responsible for populating the FIFOs for the DMAs just before they run out. By never allowing them to empty, the pulse will be repeated with zero bubble cycles in between and will repeat until the sequencer is stopped.

In [1]:
from acadia.system import Acadia

No module named 'pyxrfdc'
No module named 'pyxrfclk'


In [2]:
acadia = Acadia()

pulse_time = 100e-9

pulse_channel = acadia.DAC(0)
pulse_length = pulse_channel.seconds_to_samples(pulse_time)
pulse_memory = acadia.DACArray[pulse_channel.num](size=pulse_channel.seconds_to_bytes(pulse_time))

# Create a sequence for the sequencer
@acadia.sequence
def sequence(a):
    with a.sequencer() as seq:
        with seq.loop():
            with a.synchronizer(block=False):
                a.generate(pulse_channel, pulse_memory)
                a.generate(pulse_channel, pulse_memory)

            with seq.repeat_until(a.all_channel_fifos_empty(pulse_channel)):
                pass

def program():   
    import numpy as np
    import time
    
    # Load a constant pulse into DAC memory
    pulse = np.ones(pulse_length, dtype=np.complex64)
    pulse_samples = pulse_channel.to_samples(pulse)
    acadia.memcpy(pulse_samples, pulse_memory)
    
    # Set up the channel properties
    pulse_channel.set_nyquist_zone(2)
    pulse_channel.configure_nco(frequency=1205e6)
    pulse_channel.set_vop(4000)

    # Configure the ADC switch
    acadia.configure()

    # Reset and run the sequencer
    acadia.sequencer_reset()
    acadia.sequencer_run(sequence)
    time.sleep(0.1)
    

In [3]:
acadia.compile_all()

In [4]:
for idx,instr in enumerate(acadia._sequencer_type.instances[0]._compiled_program):
    print(f"{idx:04X}: {instr.assemble():032X} | {instr.pprint()}")

0000: 00000028283028000000000000000000 | 00000000 -> BUS_ADDR  |  DACDMAInstruction @ 00000000 -> BUS_DATA  ; Add descriptor with parameters {'trace_address': 0, 'trace_length': 25, 'fixed': False, 'decimate': 0, 'blank': False} to FIFO for DAC0
0001: 00000028283028000000000000000000 | 00000000 -> BUS_ADDR  |  DACDMAInstruction @ 00000000 -> BUS_DATA  ; Add descriptor with parameters {'trace_address': 0, 'trace_length': 25, 'fixed': False, 'decimate': 0, 'blank': False} to FIFO for DAC0
0002: 0000002828302800000C000000000001 | 000C0000 -> BUS_ADDR  |  00000001 -> BUS_DATA  ; Trigger DMAs
0003: 00000028001000000000000100000000 | 00000001 -> MASK  |  NOP
0004: 0000002800300000000C800000000000 | 000C8000 -> BUS_ADDR  |  NOP
0005: 00000000000000000000000000000000 | NOP  |  NOP  ; Latency for bus read from address 819200
0006: 00000000000000000000000000000000 | NOP  |  NOP  ; Latency for bus read from address 819200
0007: 00000000000000000000000000000000 | NOP  |  NOP  ; Latency for bus rea

In [6]:
acadia.attach()
acadia.assemble(load=True)

In [ ]:
program()

In [ ]:
acadia.sequencer_halt()
    